# Projet Santé Mentale des Adolescents

Ce notebook crée un dashboard Excel complet pour analyser l'impact des réseaux sociaux sur la santé mentale des adolescents.
Le projet utilise openpyxl pour générer les feuilles Excel avec des formules dynamiques.

## Installation des dépendances

In [80]:
%pip install openpyxl==3.1.3 pandas s3fs -q

Note: you may need to restart the kernel to use updated packages.


## Importation des packages

In [81]:
import pandas as pd
import os
from openpyxl import load_workbook, Workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.chart import BarChart, PieChart, LineChart, Reference
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo

print("Imports réussis")

Imports réussis


## Importation des données

Chargement du fichier CSV contenant les données sur la santé mentale des adolescents.

In [82]:
df = pd.read_csv('https://minio.lab.sspcloud.fr/nerojeni10/DATA_PROJET_SMA/Teen_Mental_Health_Dataset.csv')

print(f"Dimension du dataset : {df.shape}")
print(f"\nColonnes : {df.columns.tolist()}")
print(f"\nValeurs manquantes : {df.isnull().sum().sum()}")

Dimension du dataset : (1200, 13)

Colonnes : ['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label']

Valeurs manquantes : 0


## Création du fichier Excel de base

Cette étape crée le fichier Excel et y ajoute la feuille DATA avec toutes les données brutes.

In [83]:
path_file = "Projet_ODD_SIVARAJAH.xlsx"

# Créer ou récupérer le workbook
try:
    wb = load_workbook(path_file)
except:
    wb = Workbook()

# Créer le fichier s'il n'existe pas
if not os.path.exists(path_file):
    wb = Workbook()
    wb.save(path_file)

# Ajouter la feuille DATA
with pd.ExcelWriter(path_file, mode="a", if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name='DATA', index=False)

print(f"Feuilles présentes : {load_workbook(path_file).sheetnames}")

Feuilles présentes : ['Sheet', 'DATA']


## Calcul des longueurs des valeurs uniques

Avant de créer les formules, on calcule combien de valeurs uniques existent pour chaque dimension.
Ces valeurs serviront à déterminer la taille des plages dans les formules.

In [84]:
# Calculer le nombre de valeurs uniques pour chaque colonne
cols_to_calculate = ['age', 'gender', 'platform_usage', 'social_interaction_level']
len_dict = {}

for col in cols_to_calculate:
    unique_count = len(df[col].unique())
    len_dict[f"len_{col}"] = unique_count + 1

print(f"Dimensions calculées : {len_dict}")

Dimensions calculées : {'len_age': 8, 'len_gender': 3, 'len_platform_usage': 4, 'len_social_interaction_level': 4}


## Création de la feuille CALC

La feuille CALC contient les valeurs uniques pour chaque dimension. Ces valeurs sont récupérées via des formules UNIQUE.
La feuille CALC servira de base pour les filtres et les tableaux croisés.

In [85]:
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.worksheet.table import Table, TableStyleInfo

wb = load_workbook(path_file)

if "CALC" not in wb.sheetnames:
    ws_calc = wb.create_sheet("CALC")
else:
    ws_calc = wb["CALC"]

style = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)

# Genres - colonne A
formula = "=_xlfn.UNIQUE(DATA!B:B)"
ws_calc['A1'] = ArrayFormula(
    f"A1:A{len_dict['len_gender']}",
    formula
)
table_gender = Table(displayName="tblGenres", ref=f"A1:A{len_dict['len_gender']}")
table_gender.tableStyleInfo = style
table_gender.hasHeader = False
ws_calc.add_table(table_gender)

# Ages - colonne C
formula = "=_xlfn.UNIQUE(DATA!A:A)"
ws_calc['C1'] = ArrayFormula(
    f"C1:C{len_dict['len_age']}",
    formula
)
table_age = Table(displayName="tblAges", ref=f"C1:C{len_dict['len_age']}")
table_age.tableStyleInfo = style
table_age.hasHeader = False
ws_calc.add_table(table_age)

# Plateformes - colonne E
formula = "=_xlfn.UNIQUE(DATA!D:D)"
ws_calc['E1'] = ArrayFormula(
    f"E1:E{len_dict['len_platform_usage']}",
    formula
)
table_platform = Table(displayName="tblPlateformes", ref=f"E1:E{len_dict['len_platform_usage']}")
table_platform.tableStyleInfo = style
table_platform.hasHeader = False
ws_calc.add_table(table_platform)

# Interactions sociales - colonne G
formula = "=_xlfn.UNIQUE(DATA!I:I)"
ws_calc['G1'] = ArrayFormula(
    f"G1:G{len_dict['len_social_interaction_level']}",
    formula
)
table_interaction = Table(displayName="tblInteractions", ref=f"G1:G{len_dict['len_social_interaction_level']}")
table_interaction.tableStyleInfo = style
table_interaction.hasHeader = False
ws_calc.add_table(table_interaction)

wb.save(path_file)
wb.close()
print("Feuilles presentes :", load_workbook(path_file).sheetnames)

/opt/python/lib/python3.13/site-packages/openpyxl/worksheet/_writer.py:274: UserWarning: File may not be readable: column headings must be strings.
  warn("File may not be readable: column headings must be strings.")


Feuilles presentes : ['Sheet', 'DATA', 'CALC']


## Création du Tableau Croisé Dynamique

Le TCD croise deux dimensions : genres (lignes) et âges (colonnes).
Les cellules contiennent des formules SOMME.SI.ENS qui récupèrent les données depuis la feuille DATA.

In [86]:
"""
TCD CORRIGÉ - Adapté avec la formule testée dans Excel
Formule de base : =SOMME.SI.ENS(DATA!M:M;DATA!B:B;calc_sheet!A3;DATA!A:A;CALC!C2)
"""

from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

path_file = "Projet_ODD_SIVARAJAH.xlsx"

wb = load_workbook(path_file)

# Créer le TCD
if "TAB_CROISE_DYNAMIQUE" in wb.sheetnames:
    del wb["TAB_CROISE_DYNAMIQUE"]

ws_tcd = wb.create_sheet("TAB_CROISE_DYNAMIQUE")

# Styles
header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
row_header_fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid")
row_header_font = Font(bold=True)
title_fill = PatternFill(start_color="70AD47", end_color="70AD47", fill_type="solid")
title_font = Font(bold=True, size=12, color="FFFFFF")
border = Border(
    left=Side(style='thin'),
    right=Side(style='thin'),
    top=Side(style='thin'),
    bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

# Titre
ws_tcd['A1'] = "Dépression par Âge et Genre"
ws_tcd['A1'].font = title_font
ws_tcd['A1'].fill = title_fill
ws_tcd['A1'].alignment = center_align
ws_tcd.merge_cells('A1:H1')

# En-tête colonne (Genre)
ws_tcd['A2'] = "Genre"
ws_tcd['A2'].fill = header_fill
ws_tcd['A2'].font = header_font
ws_tcd['A2'].border = border
ws_tcd['A2'].alignment = center_align

# En-têtes colonnes (Âges depuis CALC)
for col_idx in range(1, 8):
    col_letter = get_column_letter(col_idx + 1)
    
    # Récupérer l'âge depuis CALC
    ws_tcd[f'{col_letter}2'] = f'=IFERROR(INDEX(CALC!$C$2:$C$1000,{col_idx}),"")'
    ws_tcd[f'{col_letter}2'].fill = header_fill
    ws_tcd[f'{col_letter}2'].font = header_font
    ws_tcd[f'{col_letter}2'].border = border
    ws_tcd[f'{col_letter}2'].alignment = center_align
    ws_tcd[f'{col_letter}2'].number_format = '0'

# En-têtes lignes (Genres depuis CALC)
for row_idx in range(1, 5):
    row_num = 2 + row_idx
    
    # Récupérer le genre depuis CALC
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$A$2:$A$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align

# Remplir les cellules avec la formule corrigée
# En ligne : Genres (A3, A4, A5, A6)
# En colonne : Âges (C2, D2, E2, F2, G2, H2, I2)
for row_idx in range(1, 5):
    row_num = 2 + row_idx  # 3, 4, 5, 6
    
    for col_idx in range(1, 8):
        col_letter = get_column_letter(col_idx + 1)  # B, C, D, E, F, G, H
        
        # Formule adaptée de celle qui fonctionne dans Excel
        # =SOMME.SI.ENS(DATA!M:M;DATA!B:B;calc_sheet!A3;DATA!A:A;CALC!C2)
        # Adaptation pour virgules (Onyxia) et références dynamiques
        formula = f'=IFERROR(SUMIFS(DATA!$M:$M,DATA!$B:$B,$A{row_num},DATA!$A:$A,CALC!${col_letter}$2),0)'
        
        ws_tcd[f'{col_letter}{row_num}'] = formula
        ws_tcd[f'{col_letter}{row_num}'].border = border
        ws_tcd[f'{col_letter}{row_num}'].alignment = center_align
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0'

# Largeurs
ws_tcd.column_dimensions['A'].width = 18
for col_idx in range(1, 8):
    ws_tcd.column_dimensions[get_column_letter(col_idx + 1)].width = 12

wb.save(path_file)
wb.close()

print("TCD créée avec formule adaptée")
print("Formule utilisée : =IFERROR(SUMIFS(DATA!$M:$M,DATA!$B:$B,$A3,DATA!$A:$A,CALC!$C$2),0)")
print("\nStructure :")
print("  - En ligne : Genres (A3, A4, A5, A6)")
print("  - En colonne : Âges (C2, D2, E2, F2, G2, H2, I2)")
print("  - Valeurs : Somme de depression (M:M)")

TCD créée avec formule adaptée
Formule utilisée : =IFERROR(SUMIFS(DATA!$M:$M,DATA!$B:$B,$A3,DATA!$A:$A,CALC!$C$2),0)

Structure :
  - En ligne : Genres (A3, A4, A5, A6)
  - En colonne : Âges (C2, D2, E2, F2, G2, H2, I2)
  - Valeurs : Somme de depression (M:M)


## Fonction pour ajouter un filtre

Cette fonction centralise la création des filtres avec validation de données.

In [87]:
def add_filter(worksheet, title_col, title_row, title_text, value_col, value_row,
               data_source_col, len_data, default_value=None, color='00C0C0C0'):
    """
    Ajoute un filtre avec dropdown à un worksheet.

    Arguments:
        worksheet: Feuille Excel cible
        title_col: Colonne du titre (ex: 'A')
        title_row: Ligne du titre (ex: 1)
        title_text: Texte du titre (ex: 'Genre')
        value_col: Colonne de la valeur/dropdown (ex: 'C')
        value_row: Ligne de la valeur
        data_source_col: Colonne source en CALC (ex: 'A' pour Genre)
        len_data: Nombre de valeurs uniques
        default_value: Valeur par défaut du dropdown
        color: Couleur de fond en hex
    """

    fill = PatternFill(start_color=color, end_color=color, fill_type='solid')
    alignment = Alignment(horizontal='center', vertical='center')
    border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )

    # Ajouter le titre
    title_cell = worksheet[f'{title_col}{title_row}']
    title_cell.value = title_text
    title_cell.alignment = alignment
    title_cell.fill = fill
    title_cell.border = border
    title_cell.font = Font(bold=True)

    # Fusionner les cellules (2x2)
    end_col = chr(ord(title_col) + 1)
    end_row = title_row + 1
    worksheet.merge_cells(f'{title_col}{title_row}:{end_col}{end_row}')

    # Ajouter la cellule de valeur avec dropdown
    value_cell = worksheet[f'{value_col}{value_row}']
    value_cell.value = default_value
    value_cell.alignment = alignment
    value_cell.fill = fill
    value_cell.border = border

    # Créer la liste de validation
    formula = f"=CALC!${data_source_col}$2:${data_source_col}${len_data}"
    dv = DataValidation(type='list', formula1=formula)
    worksheet.add_data_validation(dv)
    dv.add(f'{value_col}{value_row}')

print("Fonction add_filter définie")

Fonction add_filter définie


## Création du Dashboard TDB_1

Le dashboard TDB_1 affiche le tableau croisé avec des filtres pour sélectionner l'âge et le genre.

In [88]:
wb = load_workbook(path_file)

if "TDB_1" not in wb.sheetnames:
    TDB1_sheet = wb.create_sheet("TDB_1")
else:
    TDB1_sheet = wb["TDB_1"]

TDB1_sheet.sheet_view.showGridLines = False

# Ajouter les filtres
add_filter(TDB1_sheet, 'A', 1, 'Genre', 'C', 1, 'A', len_dict['len_gender'],
           default_value='female')
add_filter(TDB1_sheet, 'A', 4, 'Âge', 'C', 4, 'C', len_dict['len_age'],
           default_value='18')
add_filter(TDB1_sheet, 'A', 7, 'Plateforme', 'C', 7, 'E', len_dict['len_platform_usage'],
           default_value='Instagram')
add_filter(TDB1_sheet, 'A', 10, 'Interaction', 'C', 10, 'G', len_dict['len_social_interaction_level'],
           default_value='High')

wb.save(path_file)
print("Dashboard TDB_1 créé avec filtres")

Dashboard TDB_1 créé avec filtres


## Création du Dashboard TDB_2

Le dashboard TDB_2 affiche des graphiques et analyses avancées.

In [89]:
if "TDB_2" not in wb.sheetnames:
    TDB2_sheet = wb.create_sheet("TDB_2")
else:
    TDB2_sheet = wb["TDB_2"]

TDB2_sheet.sheet_view.showGridLines = False

# Ajouter les filtres
add_filter(TDB2_sheet, 'A', 1, 'Genre', 'C', 1, 'A', len_dict['len_gender'],
           default_value='female')
add_filter(TDB2_sheet, 'A', 4, 'Âge', 'C', 4, 'C', len_dict['len_age'],
           default_value='18')

wb.save(path_file)
wb.close()

print("Dashboard TDB_2 créé")

Dashboard TDB_2 créé


## Vérification finale

Afficher les feuilles créées dans le fichier Excel.

In [90]:
wb = load_workbook(path_file)
print("Feuilles créées :")
for i, sheet in enumerate(wb.sheetnames, 1):
    print(f"  {i}. {sheet}")
wb.close()

print("\nLe projet est terminé. Le fichier Excel est prêt à être utilisé.")

Feuilles créées :
  1. Sheet
  2. DATA
  3. CALC
  4. TAB_CROISE_DYNAMIQUE
  5. TDB_1
  6. TDB_2

Le projet est terminé. Le fichier Excel est prêt à être utilisé.
